# DefiLlama 
- TVL: https://defillama.com/protocol/aave?denomination=ETH

In [1]:
import requests
import pandas as pd
import os
from datetime import datetime
from io import StringIO
import json

In [2]:
def get_list(url: str, types: str = None):
    response = requests.get(url)
    data = response.json()
    df = pd.DataFrame(data)
    if types == "protocol":
        df["protocol"] = df["name"].apply(
            lambda x: x.lower().replace(" ", "-"))
    df.to_excel(f"{types}.xlsx", index=False)
    print(df.shape, df.columns)
    return df

In [3]:
def request_csv(url: str):
    response = requests.get(url, allow_redirects=True)
    data = response.content.decode("utf-8")
    if data == "Internal server error":
        return None
    return pd.read_csv(StringIO(data), sep=",")

# 1. Download TVL

### 1-1. Protocol

In [4]:
# https://api-docs.defillama.com/#tag/tvl/get/protocols
url = "https://api.llama.fi/protocols"


df_protocol = get_list(url, "protocol")


df_protocol.head()

(6341, 54) Index(['id', 'name', 'address', 'symbol', 'url', 'description', 'chain',
       'logo', 'audits', 'audit_note', 'gecko_id', 'cmcId', 'category',
       'chains', 'module', 'twitter', 'forkedFrom', 'listedAt', 'methodology',
       'misrepresentedTokens', 'slug', 'tvl', 'chainTvls', 'change_1h',
       'change_1d', 'change_7d', 'tokenBreakdowns', 'mcap', 'oraclesBreakdown',
       'audit_links', 'parentProtocol', 'wrongLiquidity', 'hallmarks',
       'parentProtocolSlug', 'referralUrl', 'treasury', 'openSource',
       'governanceID', 'github', 'staking', 'assetToken',
       'tokensExcludedFromParent', 'pool2', 'forkedFromIds', 'previousNames',
       'oracles', 'tags', 'stablecoins', 'language', 'warningBanners',
       'deadUrl', 'rugged', 'deprecated', 'protocol'],
      dtype='object')


,id,name,address,symbol,url,description,chain,logo,audits,audit_note,...,previousNames,oracles,tags,stablecoins,language,warningBanners,deadUrl,rugged,deprecated,protocol
0,2269,Binance CEX,None,-,https://www.binance.com,Binance is a cryptocurrency exchange which is ...,Multi-Chain,https://icons.llama.fi/binance-cex.jpg,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,binance-cex
1,1599,Aave V3,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,https://aave.com,"Earn interest, borrow assets, and build applic...",Multi-Chain,https://icons.llama.fi/aave-v3.png,2,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,aave-v3
2,182,Lido,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,https://lido.fi/,Liquid staking for Ethereum and Polygon. Daily...,Multi-Chain,https://icons.llama.fi/lido.png,2,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,lido
3,2272,OKX,None,-,https://www.okx.com,"OKX, formerly known as OKEx, is a Seychelles-b...",Multi-Chain,https://icons.llama.fi/okx.jpg,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,okx
4,2275,Bitfinex,None,-,https://www.bitfinex.com,Bitfinex facilitates a graphical trading exper...,Multi-Chain,https://icons.llama.fi/bitfinex.png,0,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,bitfinex


In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.DtypeWarning)

df_protocol = pd.read_excel("protocol.xlsx")

failed = []
os.makedirs("df_protocol", exist_ok=True)

for v in df_protocol["protocol"]:
    url = "https://api.llama.fi/dataset/%s.csv" % v
    df = request_csv(url)
    if df is None or df.empty:
        failed.append(v)
        print(v, "No data")
        continue

    try:
        df = df.reset_index()
        total = df.iloc[1].astype(str).str.contains("Total")
        tvl = df.iloc[2].astype(str).str.contains("TVL")
        col = df.columns[total & tvl]

        if len(col) == 0:
            print(v, "Columns not found")
            continue
        if len(col) > 1:
            print(v, "Multiple columns found")
            print(col)
            continue

        df = df.iloc[4:]
        df["Timestamp"] = df["Timestamp"].apply(
            lambda x: datetime.fromtimestamp(x))
        col = col.item()
        df = df[["Date", "Timestamp", col]]
        df.to_csv(f"df_protocol/{v}.csv", index=False)
    except Exception as e:
        print(v, e)

failed

### 1-2 Download Chain

In [5]:
url = "https://api.llama.fi/v2/chains"
df_chain = get_list(url, "chain")
df_chain.head()

(298, 6) Index(['gecko_id', 'tvl', 'tokenSymbol', 'cmcId', 'name', 'chainId'], dtype='object')


,gecko_id,tvl,tokenSymbol,cmcId,name,chainId
0,harmony,1.645639e+06,ONE,3945,Harmony,1666600000
1,mantle,4.591852e+08,MNT,27075,Mantle,5000
2,binancecoin,4.226293e+09,BNB,1839,BSC,56
3,aurora-near,1.615444e+07,AURORA,14803,Aurora,1313161554
4,x-layer,9.810801e+06,None,None,X Layer,None


In [ ]:
failed = []
os.makedirs("df_chain", exist_ok=True)

for name in df_chain["name"]:
    name = name.replace(" ", "_")
    try:
        url = "https://api.llama.fi/simpleChainDataset/%s?" % name
        df = request_csv(url)
        if df is None or df.empty:
            failed.append(name)
            continue

        df = df.head(1).T.iloc[1:].reset_index()
        df["date"] = df["index"].map(
            lambda x: datetime.strptime(x, '%d/%m/%Y'))
        df["date"] = df["date"].dt.strftime('%Y-%m-%d')
        df[name] = df[0]
        df = df[["date", name]]
        df.to_csv(f"df_chain/{name}.csv", index=False)

    except Exception as e:
        print(name, e)

### 1-3. Remove Zero TVL data

In [ ]:
for v in os.listdir("df_protocol"):
    df = pd.read_csv(f"df_protocol/{v}")
    df = df.iloc[:, 2:].fillna(0)
    try:
        if df.sum().sum() == 0:
            os.remove(f"df_protocol/{v}")
    except Exception as e:
        print(v, e)

for v in os.listdir("df_chain"):
    df = pd.read_csv(f"df_chain/{v}")
    df = df.iloc[:, 2:].fillna(0)
    try:
        if df.sum().sum() == 0:
            os.remove(f"df_chain/{v}")
    except Exception as e:
        print(v, e)

In [ ]:
len(os.listdir("df_chain"))

297

In [ ]:
len(os.listdir("df_protocol"))

4643

# 2. Merge

In [2]:
import os
import polars as pl
from datetime import datetime, timedelta

DATE_FORMAT = "%Y-%m-%d %H:%M:%S"


def leave_first_last(
    df: pl.DataFrame, col: str
) -> pl.DataFrame:
    df = df.with_columns([
        pl.arange(0, df.shape[0], eager=True).alias("index")
    ])
    return df.filter(
        (pl.col(col) != pl.col(col).shift(1)) |
        (pl.col(col) != pl.col(col).shift(-1)) |
        (pl.col("index") == 0) |  # 첫 번째 행 포함
        (pl.col("index") == (df.shape[0] - 1))  # 마지막 행 포함
    ).drop("index")


def drop_null_rows(
    df: pl.DataFrame
) -> pl.DataFrame:
    condition = pl.lit(False)
    for col in [col for col in df.columns if col not in ['date', 'hourly_date', 'Timestamp']]:
        condition = condition | pl.col(col).is_not_null()
    return df.filter(condition)


def change_hourly(
    df: pl.DataFrame, col: str = "hourly_date"
) -> pl.DataFrame:
    return df.group_by(col).agg([
        pl.col(company).mean().alias(company)
        for company in df.columns if company != col
    ]).sort(col)


def cast_column_type_without_date(  # skip 'Timestamp' column by starting from 1
    df: pl.DataFrame, dtype: object = float, start_colnum: int = 1
) -> pl.DataFrame:
    for col in df.columns[start_colnum:]:
        df = df.with_columns(pl.col(col).cast(dtype))
    return df


def cast_datetime(
    df: pl.DataFrame, has_ms=False, dt_format: str = None, col: str = "date"
) -> pl.DataFrame:
    return df.with_columns([
        pl.col(col).str.to_datetime(
            format=(dt_format if dt_format else (
                "%Y-%m-%dT%H:%M:%S.%f" if has_ms else DATE_FORMAT)))
    ])


def truncate_datetime(
    df: pl.DataFrame, col: str = "date", trun: str = "1h"
) -> pl.DataFrame:
    return df.with_columns(pl.col(col).dt.truncate(trun))

In [6]:
def merge_files(folder_name, STEP=1000) -> None:
    files = os.listdir(folder_name)
    results = []
    os.makedirs(f"result_{folder_name}", exist_ok=True)
    print("Total ephocs:", len(files)//STEP + 1)
    for i in range(0, len(files), STEP):
        for d in files[i:i+STEP]:
            df = pl.read_csv(
                f"{folder_name}/{d}").sort("Timestamp").drop(["Date"])
            if df.shape[1] > 2:
                print(d, df.shape)
            df = cast_column_type_without_date(
                df).rename({df.columns[1]: "tvl"})
            df = cast_datetime(df, has_ms=False, col="Timestamp")
            df = truncate_datetime(df, "Timestamp", "1d")
            df = df.with_columns(pl.lit(d.replace(".csv", "")).alias("name"))
            results.append(df)

        result = pl.concat(results, how="vertical").sort("Timestamp")
        fpath = f"result_{folder_name}/{STEP}_{i}.csv"
        result.write_csv(fpath)
        print(fpath, result.shape)


merge_files("df_protocol", STEP=1000)
merge_files("df_chain", STEP=1000)